In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import sklearn
sklearn.set_config(transform_output="pandas") # Forces all transformers to output DataFrames!

# Data Loading & Splitting
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, OrdinalEncoder

# Pipelines and Transformers
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Modeling & Evaluation
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV, Ridge



import warnings
warnings.filterwarnings('ignore')

In [2]:
rental = pd.read_csv('./data/data.csv')
rental.head()

,Property_id,Offer,URL,Property_type,Include_w_e,Title,Area,Governorate,Beds,Baths,Size,Availability_date,Agent_name,Agency,Amenities,rent
0,6046,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Inclusive,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3+ Maid,4,"2,368 sqft / 220 sqm",9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0
1,2240,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6,"3,229 sqft / 300 sqm",16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0
2,6248,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Inclusive,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,studio,1,484 sqft / 45 sqm,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0
3,7177,Rent,https://www.propertyfinder.bh/en/plp/rent/apar...,Apartment,Exclusive,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3,"1,507 sqft / 140 sqm",6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0
4,7842,Rent,https://www.propertyfinder.bh/en/plp/rent/vill...,Villa,Exclusive,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5+ Maid,5,"4,844 sqft / 450 sqm",30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0


In [3]:
rental.info()

<class 'pandas.DataFrame'>
RangeIndex: 10578 entries, 0 to 10577
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Property_id        10578 non-null  int64  
 1   Offer              10578 non-null  str    
 2   URL                10578 non-null  str    
 3   Property_type      10578 non-null  str    
 4   Include_w_e        10578 non-null  str    
 5   Title              10575 non-null  str    
 6   Area               10575 non-null  str    
 7   Governorate        10575 non-null  str    
 8   Beds               10578 non-null  str    
 9   Baths              10578 non-null  str    
 10  Size               10578 non-null  str    
 11  Availability_date  10059 non-null  str    
 12  Agent_name         10575 non-null  str    
 13  Agency             10575 non-null  str    
 14  Amenities          10272 non-null  float64
 15  rent               10575 non-null  float64
dtypes: float64(2), int64(1), str(13)


In [4]:
rental.describe()

,Property_id,Amenities,rent
count,10578.000000,10272.000000,10575.000000
mean,7059.734260,10.319996,718.750071
std,4080.910374,5.093414,5318.027519
min,2.000000,1.000000,50.000000
25%,3521.250000,6.000000,330.000000
50%,7023.500000,11.000000,450.000000
75%,10605.750000,14.000000,700.000000
max,14103.000000,25.000000,400000.000000


In [5]:
print("Beds unique:", rental['Beds'].unique())
print("Baths unique:", rental['Baths'].unique())
print("Size samples:", rental['Size'].head(10))
print("Offer unique:", rental['Offer'].unique())
print("Property_type unique:", rental['Property_type'].unique())
print("Include_w_e unique:", rental['Include_w_e'].unique())
print("Governorate unique:", rental['Governorate'].unique())
print("Area nunique:", rental['Area'].nunique())

Beds unique: <ArrowStringArray>
[     '3+ Maid',            '5',       'studio',            '3',
      '5+ Maid',            '2',      '4+ Maid',            '1',
      '2+ Maid',            '4',            '6',      '1+ Maid',
           '7+',      '6+ Maid', 'studio+ Maid',            '0',
     '7++ Maid',            '7',      '7+ Maid']
Length: 19, dtype: str
Baths unique: <ArrowStringArray>
['4', '6', '1', '3', '5', '2', '7+', '7', 'none']
Length: 9, dtype: str
Size samples: 0    2,368 sqft / 220 sqm
1    3,229 sqft / 300 sqm
2       484 sqft / 45 sqm
3    1,507 sqft / 140 sqm
4    4,844 sqft / 450 sqm
5    1,238 sqft / 115 sqm
6    3,014 sqft / 280 sqm
7    4,898 sqft / 455 sqm
8    1,615 sqft / 150 sqm
9     1,001 sqft / 93 sqm
Name: Size, dtype: str
Offer unique: <ArrowStringArray>
['Rent']
Length: 1, dtype: str
Property_type unique: <ArrowStringArray>
[                       'Villa',                    'Apartment',
                       'Duplex',                    'Penthouse',

In [6]:
rental.isnull().sum()

Property_id            0
Offer                  0
URL                    0
Property_type          0
Include_w_e            0
Title                  3
Area                   3
Governorate            3
Beds                   0
Baths                  0
Size                   0
Availability_date    519
Agent_name             3
Agency                 3
Amenities            306
rent                   3
dtype: int64

In [7]:
rental_copy = rental.copy()

In [8]:
rental_copy = rental_copy.dropna(subset=['rent'])

In [9]:
rental_copy.isnull().sum()

Property_id            0
Offer                  0
URL                    0
Property_type          0
Include_w_e            0
Title                  0
Area                   0
Governorate            0
Beds                   0
Baths                  0
Size                   0
Availability_date    516
Agent_name             0
Agency                 0
Amenities            306
rent                   0
dtype: int64

In [10]:
rental_copy = rental_copy.drop(columns=['Offer','URL'])

In [11]:
rental_copy.columns

Index(['Property_id', 'Property_type', 'Include_w_e', 'Title', 'Area',
       'Governorate', 'Beds', 'Baths', 'Size', 'Availability_date',
       'Agent_name', 'Agency', 'Amenities', 'rent'],
      dtype='str')

In [12]:
rental_copy['has_maid'] = rental_copy['Beds'].str.contains('Maid', case=False, na=False).astype(int)

In [13]:
rental_copy.head()

,Property_id,Property_type,Include_w_e,Title,Area,Governorate,Beds,Baths,Size,Availability_date,Agent_name,Agency,Amenities,rent,has_maid
0,6046,Villa,Inclusive,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3+ Maid,4,"2,368 sqft / 220 sqm",9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0,1
1,2240,Villa,Exclusive,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6,"3,229 sqft / 300 sqm",16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0,0
2,6248,Apartment,Inclusive,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,studio,1,484 sqft / 45 sqm,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0,0
3,7177,Apartment,Exclusive,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3,"1,507 sqft / 140 sqm",6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0,0
4,7842,Villa,Exclusive,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5+ Maid,5,"4,844 sqft / 450 sqm",30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0,1


In [14]:
rental_copy['Beds'] = rental_copy['Beds'].str.replace('Maid','',case=False)
rental_copy['Beds'] = rental_copy['Beds'].str.replace('+','',case=False)
rental_copy['Beds'] = rental_copy['Beds'].str.strip()
rental_copy['Beds'] = rental_copy['Beds'].replace('studio', '0')
rental_copy['Beds'] = pd.to_numeric(rental_copy['Beds'])

In [15]:
rental_copy['Beds'].value_counts()

Beds
2    4154
1    2250
3    2080
4    1068
0     607
5     316
6      61
7      39
Name: count, dtype: int64

In [16]:
rental_copy['Baths'].value_counts()

Baths
2       4259
3       2427
1       1583
4       1217
5        739
6        222
7         83
7+        44
none       1
Name: count, dtype: int64

In [17]:
rental_copy['Baths'] = rental_copy['Baths'].str.replace('+','')
rental_copy['Baths'] = rental_copy['Baths'].replace('none', np.nan)
rental_copy['Baths'] = pd.to_numeric(rental_copy['Baths'])

In [18]:
rental_copy['Baths'].value_counts()

Baths
2.0    4259
3.0    2427
1.0    1583
4.0    1217
5.0     739
6.0     222
7.0     127
Name: count, dtype: int64

In [19]:
rental_copy.info()

<class 'pandas.DataFrame'>
Index: 10575 entries, 0 to 10577
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Property_id        10575 non-null  int64  
 1   Property_type      10575 non-null  str    
 2   Include_w_e        10575 non-null  str    
 3   Title              10575 non-null  str    
 4   Area               10575 non-null  str    
 5   Governorate        10575 non-null  str    
 6   Beds               10575 non-null  int64  
 7   Baths              10574 non-null  float64
 8   Size               10575 non-null  str    
 9   Availability_date  10059 non-null  str    
 10  Agent_name         10575 non-null  str    
 11  Agency             10575 non-null  str    
 12  Amenities          10269 non-null  float64
 13  rent               10575 non-null  float64
 14  has_maid           10575 non-null  int64  
dtypes: float64(3), int64(3), str(9)
memory usage: 2.9 MB


In [20]:
rental_copy.head()

,Property_id,Property_type,Include_w_e,Title,Area,Governorate,Beds,Baths,Size,Availability_date,Agent_name,Agency,Amenities,rent,has_maid
0,6046,Villa,Inclusive,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3,4.0,"2,368 sqft / 220 sqm",9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0,1
1,2240,Villa,Exclusive,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6.0,"3,229 sqft / 300 sqm",16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0,0
2,6248,Apartment,Inclusive,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,0,1.0,484 sqft / 45 sqm,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0,0
3,7177,Apartment,Exclusive,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3.0,"1,507 sqft / 140 sqm",6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0,0
4,7842,Villa,Exclusive,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5,5.0,"4,844 sqft / 450 sqm",30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0,1


In [21]:
rental_copy.isnull().sum()

Property_id            0
Property_type          0
Include_w_e            0
Title                  0
Area                   0
Governorate            0
Beds                   0
Baths                  1
Size                   0
Availability_date    516
Agent_name             0
Agency                 0
Amenities            306
rent                   0
has_maid               0
dtype: int64

In [22]:
rental_copy = rental_copy.dropna(subset=['Baths'])

In [23]:
X = rental_copy[['Beds','Baths','has_maid']]
y = rental_copy['rent']

In [24]:
rental_copy

,Property_id,Property_type,Include_w_e,Title,Area,Governorate,Beds,Baths,Size,Availability_date,Agent_name,Agency,Amenities,rent,has_maid
0,6046,Villa,Inclusive,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3,4.0,"2,368 sqft / 220 sqm",9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0,1
1,2240,Villa,Exclusive,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6.0,"3,229 sqft / 300 sqm",16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0,0
2,6248,Apartment,Inclusive,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,0,1.0,484 sqft / 45 sqm,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0,0
3,7177,Apartment,Exclusive,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3.0,"1,507 sqft / 140 sqm",6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0,0
4,7842,Villa,Exclusive,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5,5.0,"4,844 sqft / 450 sqm",30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10573,5192,Apartment,Inclusive,View Of Water / Maid Service / Stylish Furnished,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,2,3.0,"1,453 sqft / 135 sqm",29 Aug 2025,Al Ward Real Estate,AlWard Real Estate,17.0,550.0,0
10574,13419,Apartment,Exclusive,2 Bedroom Fully Furnished Flat on low price offer,Al Juffair,Capital Governorate,2,2.0,"1,292 sqft / 120 sqm",NaN,Muhammad,Al Waleed Homes,8.0,300.0,0
10575,5391,Apartment,Inclusive,Sea View | Free Internet + Housekeeping | Premium,Seef,Capital Governorate,1,2.0,980 sqft / 91 sqm,27 Aug 2025,Bilal Mohamed,River West Properties,11.0,650.0,0
10576,861,Villa,Exclusive,Zinj Standalone Villa | Private Pool &amp; Garden,"Zinj, Manama",Capital Governorate,3,4.0,"4,306 sqft / 400 sqm",24 Sep 2025,Shaji Kottathazham,AIM REAL ESTATE,11.0,950.0,1


In [25]:
rental_copy['Size_sqm'] = rental_copy['Size'].str.split('/').str[-1]
rental_copy['Size_sqm'] = rental_copy['Size_sqm'].str.replace('sqm', '', case=False)
rental_copy['Size_sqm'] = rental_copy['Size_sqm'].str.replace(',', '')
rental_copy['Size_sqm'] = pd.to_numeric(rental_copy['Size_sqm'], errors='coerce')
rental_copy = rental_copy.drop(columns=['Size'])

In [26]:
rental_copy['is_inclusive'] = (rental_copy['Include_w_e'] == 'Inclusive').astype(int)
rental_copy = rental_copy.drop(columns=['Include_w_e'])

In [27]:
rental_copy.info()

<class 'pandas.DataFrame'>
Index: 10574 entries, 0 to 10577
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Property_id        10574 non-null  int64  
 1   Property_type      10574 non-null  str    
 2   Title              10574 non-null  str    
 3   Area               10574 non-null  str    
 4   Governorate        10574 non-null  str    
 5   Beds               10574 non-null  int64  
 6   Baths              10574 non-null  float64
 7   Availability_date  10058 non-null  str    
 8   Agent_name         10574 non-null  str    
 9   Agency             10574 non-null  str    
 10  Amenities          10269 non-null  float64
 11  rent               10574 non-null  float64
 12  has_maid           10574 non-null  int64  
 13  Size_sqm           10574 non-null  int64  
 14  is_inclusive       10574 non-null  int64  
dtypes: float64(3), int64(5), str(7)
memory usage: 2.6 MB


In [28]:
rental_copy

,Property_id,Property_type,Title,Area,Governorate,Beds,Baths,Availability_date,Agent_name,Agency,Amenities,rent,has_maid,Size_sqm,is_inclusive
0,6046,Villa,BTRAND NEW LUXURY VILLA FOR RENT AMWAJ,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,3,4.0,9 Apr 2025,Savio Fernandes,Almoayyed property solutions,6.0,1000.0,1,220,1
1,2240,Villa,First Resident | Quiet Area | Luxurious,Diyar Al Muharraq,Muharraq Governorate,5,6.0,16 Sep 2025,Al Lagoon Real Estate,Al Lagoon Real Estate,5.0,700.0,0,300,0
2,6248,Apartment,Studio Apartment for rent located in Juffair,Al Juffair,Capital Governorate,0,1.0,17 Aug 2025,Najeeb Hejres,Green Line Real Estate,12.0,260.0,0,45,1
3,7177,Apartment,Apartment for Rent in Jidhafs – Prime Location,Jidhafs,Northern Governorate,3,3.0,6 Aug 2025,Sayed Sadeq Jaafar,Grnata Real Estate,2.0,250.0,0,140,0
4,7842,Villa,Navy Approved| 5 BR | Private Pool |Gardens,Al Juffair,Capital Governorate,5,5.0,30 Jul 2025,Zsuzsanna Gecse,Arabian Homes Properties,13.0,1280.0,1,450,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10573,5192,Apartment,View Of Water / Maid Service / Stylish Furnished,"Amwaj Avenue, Amwaj Islands",Muharraq Governorate,2,3.0,29 Aug 2025,Al Ward Real Estate,AlWard Real Estate,17.0,550.0,0,135,1
10574,13419,Apartment,2 Bedroom Fully Furnished Flat on low price offer,Al Juffair,Capital Governorate,2,2.0,NaN,Muhammad,Al Waleed Homes,8.0,300.0,0,120,0
10575,5391,Apartment,Sea View | Free Internet + Housekeeping | Premium,Seef,Capital Governorate,1,2.0,27 Aug 2025,Bilal Mohamed,River West Properties,11.0,650.0,0,91,1
10576,861,Villa,Zinj Standalone Villa | Private Pool &amp; Garden,"Zinj, Manama",Capital Governorate,3,4.0,24 Sep 2025,Shaji Kottathazham,AIM REAL ESTATE,11.0,950.0,1,400,0


In [29]:
rental_copy = rental_copy.drop(columns=['Property_id', 'Title', 'Availability_date', 'Agent_name', 'Agency'])

In [30]:
X = rental_copy.drop(columns=['rent'])
y = rental_copy['rent']

numeric_cols = ['Beds', 'Baths', 'Size_sqm', 'Amenities', 'has_maid', 'is_inclusive']
categorical_cols = ['Property_type', 'Governorate', 'Area']

print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)
print("X shape:", X.shape)

Numeric: ['Beds', 'Baths', 'Size_sqm', 'Amenities', 'has_maid', 'is_inclusive']
Categorical: ['Property_type', 'Governorate', 'Area']
X shape: (10574, 9)


In [31]:
rental_copy.isnull().sum()

Property_type      0
Area               0
Governorate        0
Beds               0
Baths              0
Amenities        305
rent               0
has_maid           0
Size_sqm           0
is_inclusive       0
dtype: int64

In [32]:
rental_copy.describe()

,Beds,Baths,Amenities,rent,has_maid,Size_sqm,is_inclusive
count,10574.000000,10574.000000,10269.000000,10574.000000,10574.000000,10574.000000,10574.000000
mean,2.202478,2.663703,10.321258,718.784944,0.210800,171.275392,0.612446
std,1.206382,1.290132,5.093565,5318.277795,0.407896,204.971659,0.487215
min,0.000000,1.000000,1.000000,50.000000,0.000000,1.000000,0.000000
25%,1.000000,2.000000,6.000000,330.000000,0.000000,90.000000,0.000000
50%,2.000000,2.000000,11.000000,450.000000,0.000000,125.000000,1.000000
75%,3.000000,3.000000,14.000000,700.000000,0.000000,180.000000,1.000000
max,7.000000,7.000000,25.000000,400000.000000,1.000000,13742.000000,1.000000


In [33]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first',sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LassoCV())
])

In [34]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [35]:
X = rental_copy.drop(columns=['rent'])
y = np.log1p(rental_copy['rent'])

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f} BHD")

MAE: 0.21 BHD


In [37]:
y_pred_actual_logonly = np.exp(y_pred)
y_test_actual_logonly = np.exp(y_test)

mae = mean_absolute_error(y_test_actual_logonly, y_pred_actual_logonly)
print(f"MAE: {mae:.2f} BHD")

MAE: 8633.06 BHD


In [38]:
rental_copy = rental_copy[rental_copy['rent'] <= 5000]

X = rental_copy.drop(columns=['rent'])
y = np.log1p(rental_copy['rent'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

y_pred_actual = np.exp(y_pred)
y_test_actual = np.exp(y_test)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
print(f"MAE: {mae:.2f} BHD")

MAE: 124.56 BHD


In [39]:
pipe_ridge = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RidgeCV())
])

In [40]:
rental_copy = rental_copy[rental_copy['rent'] <= 5000]

X = rental_copy.drop(columns=['rent'])
y = np.log1p(rental_copy['rent'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

pipe_ridge.fit(X_train, y_train)
y_pred = pipe_ridge.predict(X_test)

y_pred_actual = np.exp(y_pred)
y_test_actual = np.exp(y_test)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
print(f"MAE: {mae:.2f} BHD")

MAE: 113.25 BHD


In [41]:
test = pd.read_csv('./data/test.csv')

In [42]:
test_ids = test['Property_id']

test = test.drop(columns=['Offer', 'URL'])

test['has_maid'] = test['Beds'].str.contains('Maid', case=False, na=False).astype(int)
test['Beds'] = test['Beds'].str.replace('Maid', '', case=False)
test['Beds'] = test['Beds'].str.replace('+', '', case=False)
test['Beds'] = test['Beds'].str.strip()
test['Beds'] = test['Beds'].replace('studio', '0')
test['Beds'] = pd.to_numeric(test['Beds'])

test['Baths'] = test['Baths'].str.replace('+', '')
test['Baths'] = test['Baths'].replace('none', np.nan)
test['Baths'] = pd.to_numeric(test['Baths'])

test['Size_sqm'] = test['Size'].str.split('/').str[-1]
test['Size_sqm'] = test['Size_sqm'].str.replace('sqm', '', case=False)
test['Size_sqm'] = test['Size_sqm'].str.replace(',', '')
test['Size_sqm'] = pd.to_numeric(test['Size_sqm'], errors='coerce')
test = test.drop(columns=['Size'])

test['is_inclusive'] = (test['Include_w_e'] == 'Inclusive').astype(int)
test = test.drop(columns=['Include_w_e'])

test = test.drop(columns=['Property_id', 'Title', 'Availability_date', 'Agent_name', 'Agency'])

y_pred_log = pipe_ridge.predict(test)
y_pred_final = np.expm1(y_pred_log)

submission = pd.DataFrame({
    'Property_id': test_ids,
    'Rent': y_pred_final
})
submission.to_csv('submission.csv', index=False)
print(submission.head())

   Property_id        Rent
0         4394  400.033767
1         2338  410.466875
2         8531  761.874269
3         8952  345.467462
4        11064  245.440227


In [43]:
sub_final = pd.concat([test_ids, pd.Series(y_pred_final, name='rent')], axis=1)

In [44]:
sub_final

,Property_id,rent
0,4394,400.033767
1,2338,410.466875
2,8531,761.874269
3,8952,345.467462
4,11064,245.440227
...,...,...
3522,13597,279.515438
3523,2475,409.678368
3524,2685,773.217541
3525,2523,249.792675


In [46]:
test

,Property_type,Area,Governorate,Beds,Baths,Amenities,has_maid,Size_sqm,is_inclusive
0,Apartment,"Sanabis, Manama",Capital Governorate,2,2.0,11.0,0,122,1
1,Apartment,Al Juffair,Capital Governorate,2,2.0,12.0,0,180,0
2,Compound,Jannusan,Northern Governorate,3,4.0,16.0,1,350,0
3,Apartment,Saar,Northern Governorate,2,2.0,6.0,0,120,1
4,Apartment,Saar,Northern Governorate,1,1.0,5.0,0,98,0
...,...,...,...,...,...,...,...,...,...
3522,Apartment,Hidd,Muharraq Governorate,2,2.0,6.0,0,120,0
3523,Apartment,"Zinj, Manama",Capital Governorate,2,3.0,7.0,0,130,1
3524,Villa,Barbar,Northern Governorate,3,3.0,14.0,1,350,0
3525,Apartment,Galali,Muharraq Governorate,2,2.0,8.0,1,160,0


In [46]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('power', PowerTransformer())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

pipe_power = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

# Cross-validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipe_power, X, y, cv=5, scoring='neg_mean_absolute_error')
print("CV MAE scores (log):", -scores)
print("Mean CV MAE (log):", (-scores).mean())

# Fit on all training data and predict test
pipe_power.fit(X, y)
y_pred_log = pipe_power.predict(test)
y_pred_final = np.expm1(y_pred_log)

submission = pd.DataFrame({
    'Property_id': test_ids,
    'Rent': y_pred_final
})
submission.to_csv('submission_power_ridge.csv', index=False)
print(submission.head())

CV MAE scores (log): [0.19570576 0.19953167 0.19503063 0.19985375 0.19482335]
Mean CV MAE (log): 0.19698903241345947
   Property_id        Rent
0         4394  410.533431
1         2338  423.578913
2         8531  813.513775
3         8952  356.924735
4        11064  238.023550


In [48]:
from sklearn.metrics import root_mean_squared_error

mae = mean_absolute_error(y_test_actual, y_pred_actual)
rmse = root_mean_squared_error(y_test_actual, y_pred_actual)
r2 = r2_score(y_test_actual, y_pred_actual)

print(f"MAE:  {mae:.2f} BHD")
print(f"RMSE: {rmse:.2f} BHD")
print(f"R2:   {r2:.4f}")

MAE:  114.74 BHD
RMSE: 189.73 BHD
R2:   0.7229


In [49]:
test

,Property_type,Area,Governorate,Beds,Baths,Amenities,has_maid,Size_sqm,is_inclusive
0,Apartment,"Sanabis, Manama",Capital Governorate,2,2.0,11.0,0,122,1
1,Apartment,Al Juffair,Capital Governorate,2,2.0,12.0,0,180,0
2,Compound,Jannusan,Northern Governorate,3,4.0,16.0,1,350,0
3,Apartment,Saar,Northern Governorate,2,2.0,6.0,0,120,1
4,Apartment,Saar,Northern Governorate,1,1.0,5.0,0,98,0
...,...,...,...,...,...,...,...,...,...
3522,Apartment,Hidd,Muharraq Governorate,2,2.0,6.0,0,120,0
3523,Apartment,"Zinj, Manama",Capital Governorate,2,3.0,7.0,0,130,1
3524,Villa,Barbar,Northern Governorate,3,3.0,14.0,1,350,0
3525,Apartment,Galali,Muharraq Governorate,2,2.0,8.0,1,160,0
